In [9]:
import pandas as pd
import networkx as nx
from pathlib import Path
import numpy as np

# Go up one folder from code/ and into output/
output = Path.cwd().parent / "output"

edges = pd.read_csv(output / "airport_edges_2025.csv")
attrs = pd.read_csv(output / "airport_attributes_2025.csv", index_col="airport")

G = nx.from_pandas_edgelist(
    edges,
    "origin",
    "dest",
    edge_attr="passengers",
    create_using=nx.DiGraph
)

nx.set_node_attributes(G, attrs.to_dict("index"))

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

Nodes: 576
Edges: 8156


In [54]:
top_30_measles = attrs.sort_values(
    by="cases_2025",
    ascending=False
).head(30)

print(top_30_measles[["name", "state", "cases_2025"]])

                                                      name state  cases_2025
airport                                                                     
HOB                            Lea County Regional Airport    NM         501
GSP           Greenville-Spartanburg International Airport    SC         330
IFP                Laughlin Bullhead International Airport    AZ         216
SGU                             St George Regional Airport    UT         155
LBB            Lubbock Preston Smith International Airport    TX         144
ELP                          El Paso International Airport    TX          61
MAF               Midland International Air and Space Port    TX          52
GCK                           Garden City Regional Airport    KS          48
GRB                  Austin Straubel International Airport    WI          36
TYR                          Tyler Pounds Regional Airport    TX          29
PVU                                Provo Municipal Airport    UT          25

In [55]:
bc = nx.betweenness_centrality(G)

top_20_bc = sorted(bc.items(), key=lambda x: x[1], reverse=True)[:20]

bc_table = pd.DataFrame([
    {
        "Airport Code": node,
        "Airport": G.nodes[node]["name"],
        "State": G.nodes[node]["state"],
        "Betweenness Centrality": value
    }
    for node, value in top_20_bc
])

print(bc_table.to_string(index=False))

Airport Code                                                               Airport State  Betweenness Centrality
         ANC                           Ted Stevens Anchorage International Airport    AK                0.285674
         DEN                                          Denver International Airport    CO                0.147030
         ORD                                  Chicago O'Hare International Airport    IL                0.128274
         DFW                               Dallas Fort Worth International Airport    TX                0.121326
         BET                                                        Bethel Airport    AK                0.107837
         MSP Minneapolis–Saint Paul International Airport / Wold–Chamberlain Field    MN                0.058551
         ATL                      Hartsfield Jackson Atlanta International Airport    GA                0.056581
         SEA                                  Seattle–Tacoma International Airport    WA        

In [56]:
ec = nx.eigenvector_centrality(G)

top_20_ec = sorted(ec.items(), key=lambda x: x[1], reverse=True)[:20]

ec_table = pd.DataFrame([
    {
        "Airport Code": node,
        "Airport": G.nodes[node]["name"],
        "State": G.nodes[node]["state"],
        "Eigenvector Centrality": value
    }
    for node, value in top_20_ec
])

print(ec_table.to_string(index=False))

Airport Code                                                               Airport State  Eigenvector Centrality
         DFW                               Dallas Fort Worth International Airport    TX                0.169437
         ATL                      Hartsfield Jackson Atlanta International Airport    GA                0.167951
         DEN                                          Denver International Airport    CO                0.166967
         ORD                                  Chicago O'Hare International Airport    IL                0.166329
         CLT                               Charlotte Douglas International Airport    NC                0.154024
         LAS                                      Harry Reid International Airport    NV                0.152208
         MSP Minneapolis–Saint Paul International Airport / Wold–Chamberlain Field    MN                0.150450
         MCO                                         Orlando International Airport    FL        

In [57]:
cc = nx.closeness_centrality(G)

top_20_cc = sorted(cc.items(), key=lambda x: x[1], reverse=True)[:20]

cc_table = pd.DataFrame([
    {
        "Airport Code": node,
        "Airport": G.nodes[node]["name"],
        "State": G.nodes[node]["state"],
        "Closeness Centrality": value
    }
    for node, value in top_20_cc
])

print(cc_table.to_string(index=False))

Airport Code                                                               Airport State  Closeness Centrality
         DEN                                          Denver International Airport    CO              0.530708
         ORD                                  Chicago O'Hare International Airport    IL              0.528129
         DFW                               Dallas Fort Worth International Airport    TX              0.520043
         ATL                      Hartsfield Jackson Atlanta International Airport    GA              0.504592
         MSP Minneapolis–Saint Paul International Airport / Wold–Chamberlain Field    MN              0.500871
         LAS                                      Harry Reid International Airport    NV              0.499490
         PHX                              Phoenix Sky Harbor International Airport    AZ              0.489590
         SEA                                  Seattle–Tacoma International Airport    WA              0.487832
 

In [60]:
measles_nodes = set(top_30_measles.index)

bc_nodes = {node for node, value in top_20_bc}
ec_nodes = {node for node, value in top_20_ec}
cc_nodes = {node for node, value in top_20_cc}

centrality_nodes = bc_nodes | ec_nodes | cc_nodes

common_nodes = measles_nodes & centrality_nodes

for node in common_nodes:
    print(node, G.nodes[node]["name"],)

DEN Denver International Airport
ATL Hartsfield Jackson Atlanta International Airport
MSP Minneapolis–Saint Paul International Airport / Wold–Chamberlain Field
